# Fase 2 — Limpieza y Transformación (Data Cleaning)
## Dataset: ABC Corporation — HR Analytics

**Objetivo:** Convertir los datos sucios del EDA en datos limpios, correctos y listos para análisis y modelado.

**Regla de oro:** Nunca se modifica el CSV original. Todo el trabajo se hace sobre una copia.

**Pasos que se realizarán:**
1. Eliminación de columnas constantes y filas duplicadas
2. Normalización de texto en `JobRole`
3. Corrección de errata en `MaritalStatus`
4. Tratamiento de nulos categóricos (moda)
5. Tratamiento de nulos numéricos (mediana)
6. Corrección de tipos de datos (float64 → int64)
7. Codificación de variables binarias (`Attrition`, `OverTime`)
8. Pipeline completo encadenado
9. Validación de calidad (asserts)
10. Guardado del dataset limpio

---

## Librerías e importaciones

In [20]:
# Librerías necesarias para la limpieza y transformación de datos
import pandas as pd
import numpy as np

# Configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)

## Carga del dataset y creación de la copia de trabajo

In [21]:
# Cargamos el dataset original
df_original = pd.read_csv('hr.csv')
 
# Creamos una copia de trabajo para no modificar nunca el archivo original
# Regla de oro: todas las transformaciones se hacen sobre df_clean
df_clean = df_original.copy()

# Confirmamos dimensiones
print(f'Filas: {df_clean.shape[0]}')
print(f'Columnas: {df_clean.shape[1]}')
df_clean.head(3)

Filas: 1474
Columnas: 35


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41.0,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,sALES eXECUTIVE,4.0,Single,5993.0,19479,8,Y,Yes,11,3,1,80.0,0,8,0.0,1,6,4,0,5.0
1,49.0,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,rESEARCH sCIENTIST,2.0,Married,5130.0,24907,1,Y,No,23,4,4,NaN,1,10,3.0,3,10,7,1,7.0
2,37.0,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,lABORATORY tECHNICIAN,3.0,Single,2090.0,2396,6,Y,Yes,15,3,2,NaN,0,7,3.0,3,0,0,0,0.0


---
## Paso 1 — Eliminamos columnas constantes y filas duplicadas

- **Columnas constantes:** `EmployeeCount`, `Over18`, `StandardHours` (siempre el mismo valor). No aportan variabilidad.
- **Filas duplicadas:** se detectaron 4 filas duplicadas exactas en la Fase 1.

In [22]:
def drop_useless_cols(df):
    """
    Elimina las columnas constantes que no aportan
    variabilidad ni valor al análisis.
    Las detecta automáticamente en vez de usar una lista fija.
    """
    df_mod = df.copy()
    # Detectamos automáticamente las columnas con un solo valor único
    cols_constantes = [col for col in df_mod.columns if df_mod[col].nunique() == 1]
    print(f'Columnas constantes eliminadas: {cols_constantes}')
    return df_mod.drop(columns=cols_constantes, errors='ignore')


def drop_duplicates(df):
    """
    Elimina las filas duplicadas del dataset.
    En la Fase 1 detectamos 4 duplicados al final del archivo.
    """
    df_mod = df.drop_duplicates()
    return df_mod


# Aplicamos las dos funciones
df_clean = drop_useless_cols(df_clean)
df_clean = drop_duplicates(df_clean)

# Verificamos el resultado
print(f'Columnas originales: {df_original.shape[1]}')
print(f'Columnas tras limpieza: {df_clean.shape[1]}')
print(f'Filas originales: {df_original.shape[0]}')
print(f'Filas tras eliminar duplicados: {df_clean.shape[0]}')

Columnas constantes eliminadas: ['EmployeeCount', 'Over18', 'StandardHours']
Columnas originales: 35
Columnas tras limpieza: 32
Filas originales: 1474
Filas tras eliminar duplicados: 1470


---
## Paso 2 — Normalizamos el formato de JobRole

Problema: texto corrupto con espacios y mayúsculas mezcladas (ej. `' sALES eXECUTIVE '`).
Sin limpiar, Python trataría `'Sales Executive'` y `' sALES eXECUTIVE '` como dos categorías distintas.

In [23]:
def fix_job_role(df):
    """
    Elimina espacios en blanco en los extremos de 'JobRole'
    y transforma el texto a formato título estándar.
    """
    df_mod = df.copy()
    # .str.strip() elimina espacios al inicio y al final
    # .str.title() convierte a formato título (Primera Letra Mayúscula)
    df_mod['JobRole'] = df_mod['JobRole'].str.strip().str.title()
    return df_mod


# Aplicamos la función
df_clean = fix_job_role(df_clean)

# Verificamos el resultado
print('--- Valores de JobRole corregidos ---')
print(df_clean['JobRole'].unique())

--- Valores de JobRole corregidos ---
<StringArray>
[          'Sales Executive',        'Research Scientist',
     'Laboratory Technician',    'Manufacturing Director',
 'Healthcare Representative',                   'Manager',
      'Sales Representative',         'Research Director',
           'Human Resources']
Length: 9, dtype: str


**Resultado:** 9 roles con formato correcto y listos para graficar en Fase 3.

---
## Paso 3 — Corregimos la errata en MaritalStatus

Problema: el valor `'Marreid'` en vez de `'Married'` generaba dos grupos para un mismo estado civil.

In [24]:
def fix_marital_status(df):
    """
    Corrige la errata de escritura 'Marreid' pasándola a 'Married'
    en la columna MaritalStatus.
    """
    df_mod = df.copy()
    # .replace() sustituye el valor incorrecto por el correcto
    df_mod['MaritalStatus'] = df_mod['MaritalStatus'].replace({'Marreid': 'Married'})
    return df_mod


# Aplicamos la función
df_clean = fix_marital_status(df_clean)

# Verificamos el resultado
# Nota: el nan que aparece aquí se tratará en el siguiente paso
print('--- Valores de MaritalStatus corregidos ---')
print(df_clean['MaritalStatus'].unique())

--- Valores de MaritalStatus corregidos ---
<StringArray>
['Single', 'Married', 'Divorced', nan]
Length: 4, dtype: str


---
## Paso 4 — Tratamos los nulos categóricos (moda)

Las columnas categóricas con nulos se rellenan con la **moda** (el valor más frecuente).

In [25]:
def fix_categorical_nulls(df):
    """
    Rellena los valores nulos de las columnas categóricas
    con la moda (valor más frecuente) de cada columna.
    """
    df_mod = df.copy()
    # Columnas categóricas con nulos detectadas en Fase 1
    cols_with_nulls = ['Department', 'MaritalStatus', 'OverTime', 'BusinessTravel',
                       'EducationField']
    for col in cols_with_nulls:
        # .mode()[0] obtiene el valor más frecuente de la columna
        moda = df_mod[col].mode()[0]
        df_mod[col] = df_mod[col].fillna(moda)
        print(f'  {col}: nulos rellenados con moda ("{moda}")')
    return df_mod


# Aplicamos la función
df_clean = fix_categorical_nulls(df_clean)

# Verificamos que no quedan nulos en estas columnas
print('\n--- Nulos restantes en columnas categóricas ---')
print(df_clean[['Department', 'MaritalStatus', 'OverTime',
                'BusinessTravel', 'EducationField']].isnull().sum())

  Department: nulos rellenados con moda ("Research & Development")
  MaritalStatus: nulos rellenados con moda ("Married")
  OverTime: nulos rellenados con moda ("No")
  BusinessTravel: nulos rellenados con moda ("Travel_Rarely")
  EducationField: nulos rellenados con moda ("Life Sciences")

--- Nulos restantes en columnas categóricas ---
Department        0
MaritalStatus     0
OverTime          0
BusinessTravel    0
EducationField    0
dtype: int64


---
## Paso 5 — Tratamos los nulos numéricos (mediana)

Las columnas numéricas con nulos se rellenan con la **mediana**.
Usamos la mediana (no la media) porque es resistente a valores extremos (outliers).

In [26]:
def fix_numerical_nulls(df):
    """
    Rellena los valores nulos de las columnas numéricas con la mediana.
    Usamos la mediana porque es resistente a valores extremos (outliers).
    """
    df_mod = df.copy()
    # Columnas numéricas con nulos detectadas en Fase 1
    cols_with_nulls = ['Age', 'JobSatisfaction', 'MonthlyIncome',
                       'TrainingTimesLastYear', 'YearsWithCurrManager']
    for col in cols_with_nulls:
        # Calculamos la mediana sobre los valores reales (sin nulos)
        mediana = df_mod[col].median()
        df_mod[col] = df_mod[col].fillna(mediana)
        print(f'  {col}: nulos rellenados con mediana ({mediana})')
    return df_mod


# Aplicamos la función
df_clean = fix_numerical_nulls(df_clean)

# Verificamos que no quedan nulos en estas columnas
print('\n--- Nulos restantes en columnas numéricas ---')
print(df_clean[['Age', 'JobSatisfaction', 'MonthlyIncome',
                'TrainingTimesLastYear', 'YearsWithCurrManager']].isnull().sum())

  Age: nulos rellenados con mediana (36.0)
  JobSatisfaction: nulos rellenados con mediana (3.0)
  MonthlyIncome: nulos rellenados con mediana (4907.0)
  TrainingTimesLastYear: nulos rellenados con mediana (3.0)
  YearsWithCurrManager: nulos rellenados con mediana (3.0)

--- Nulos restantes en columnas numéricas ---
Age                      0
JobSatisfaction          0
MonthlyIncome            0
TrainingTimesLastYear    0
YearsWithCurrManager     0
dtype: int64


---
## Paso 6 — Corregimos los tipos de datos

Convertimos columnas numéricas que están en `float64` a `int64`.
Solo se aplica **después** de haber tratado los nulos, porque no se puede convertir `NaN` a entero.

In [27]:
def fix_dtypes(df):
    """
    Convierte columnas numéricas que están en float64 a int64,
    ya que representan valores enteros.
    Solo se aplica después de haber tratado los nulos.
    """
    df_mod = df.copy()
    # Columnas que deben ser enteras
    cols_to_fix = ['Age', 'JobSatisfaction', 'MonthlyIncome',
                   'TrainingTimesLastYear', 'YearsWithCurrManager']
    for col in cols_to_fix:
        # .astype(int) convierte float64 a int64
        df_mod[col] = df_mod[col].astype(int)
    return df_mod


# Aplicamos la función
df_clean = fix_dtypes(df_clean)

# Verificamos que los tipos son correctos ahora
print(df_clean[['Age', 'JobSatisfaction', 'MonthlyIncome',
                'TrainingTimesLastYear', 'YearsWithCurrManager']].dtypes)

Age                      int64
JobSatisfaction          int64
MonthlyIncome            int64
TrainingTimesLastYear    int64
YearsWithCurrManager     int64
dtype: object


---
## Paso 7 — Codificación de variables binarias (Yes/No → 1/0)

Las columnas `Attrition` y `OverTime` tienen valores `'Yes'`/`'No'`.
Las transformamos a 1/0 para facilitar el análisis y el modelado.
Los nulos de `OverTime` se convierten en `pd.NA` (nulo tipado) en vez de un -1 incorrecto.

In [28]:
def encode_binary(df):
    """
    Codifica las columnas binarias 'Attrition' y 'OverTime' a 1/0.
    Yes → 1, No → 0, Unknown/nulo → pd.NA (nulo tipado, no -1).
    """
    df_mod = df.copy()
    # El mapping incluye pd.NA para los valores desconocidos
    # Int64 (con mayúscula) es el tipo entero que soporta nulos en pandas
    binary_mapping = {'Yes': 1, 'No': 0, 'Unknown': pd.NA}
    df_mod['Attrition'] = df_mod['Attrition'].replace(binary_mapping).astype('Int64')
    df_mod['OverTime'] = df_mod['OverTime'].replace(binary_mapping).astype('Int64')
    return df_mod


# Aplicamos la función
df_clean = encode_binary(df_clean)

# Verificamos el resultado
print('--- Valores únicos finales de Attrition ---')
print(df_clean['Attrition'].unique())
print('\n--- Valores únicos finales de OverTime ---')
print(df_clean['OverTime'].unique())

--- Valores únicos finales de Attrition ---
<IntegerArray>
[1, 0]
Length: 2, dtype: Int64

--- Valores únicos finales de OverTime ---
<IntegerArray>
[1, 0]
Length: 2, dtype: Int64


---
## Paso 8 — Pipeline completo encadenado

Función que encadena todas las transformaciones en orden.
Esto es la base de la **ETL (Extract, Transform, Load)** de la Fase 5.

In [29]:
def limpiar_dataset(df):
    """
    Pipeline completo de limpieza y transformación.
    Encadena todas las funciones en el orden correcto.
    La salida es el dataset limpio y listo para Fase 3 y BBDD.

    IMPORTANTE: el orden es obligatorio.
    fix_dtypes() debe ir después de fix_numerical_nulls()
    encode_binary() debe ir después de fix_categorical_nulls()
    """
    df_resultado = df.copy()
    df_resultado = drop_useless_cols(df_resultado)      # 1. Eliminar columnas constantes
    df_resultado = drop_duplicates(df_resultado)         # 2. Eliminar duplicados
    df_resultado = fix_job_role(df_resultado)            # 3. Normalizar JobRole
    df_resultado = fix_marital_status(df_resultado)      # 4. Corregir errata MaritalStatus
    df_resultado = fix_categorical_nulls(df_resultado)   # 5. Nulos categóricos → moda
    df_resultado = fix_numerical_nulls(df_resultado)     # 6. Nulos numéricos → mediana
    df_resultado = fix_dtypes(df_resultado)              # 7. float64 → int64
    df_resultado = encode_binary(df_resultado)           # 8. Yes/No → 1/0
    return df_resultado


# Ejecutamos el pipeline completo desde el original
df_clean = limpiar_dataset(df_original)

Columnas constantes eliminadas: ['EmployeeCount', 'Over18', 'StandardHours']
  Department: nulos rellenados con moda ("Research & Development")
  MaritalStatus: nulos rellenados con moda ("Married")
  OverTime: nulos rellenados con moda ("No")
  BusinessTravel: nulos rellenados con moda ("Travel_Rarely")
  EducationField: nulos rellenados con moda ("Life Sciences")
  Age: nulos rellenados con mediana (36.0)
  JobSatisfaction: nulos rellenados con mediana (3.0)
  MonthlyIncome: nulos rellenados con mediana (4907.0)
  TrainingTimesLastYear: nulos rellenados con mediana (3.0)
  YearsWithCurrManager: nulos rellenados con mediana (3.0)


---
## Paso 9 — Control de calidad automático (asserts)

Verificamos que el pipeline ha funcionado correctamente.
Si algo falla, el assert lanza un error con un mensaje claro.

In [30]:
def validate(df_original, df_clean):
    """
    Comprueba que el pipeline ha funcionado correctamente.
    Lanza un error si algo no está bien.
    """
    # 1. Dimensiones del dataset
    assert df_clean.shape[1] == 32, f'Error: Se esperaban 32 columnas pero hay {df_clean.shape[1]}'
    assert df_clean.shape[0] == 1470, f'Error: Se esperaban 1470 filas pero hay {df_clean.shape[0]}'

    # 2. Nulos en columnas categóricas
    assert df_clean['Department'].isnull().sum() == 0, 'Error: Siguen quedando nulos en Department'
    assert df_clean['MaritalStatus'].isnull().sum() == 0, 'Error: Siguen quedando nulos en MaritalStatus'
    assert df_clean['BusinessTravel'].isnull().sum() == 0, 'Error: Siguen quedando nulos en BusinessTravel'
    assert df_clean['EducationField'].isnull().sum() == 0, 'Error: Siguen quedando nulos en EducationField'

    # 3. Nulos en columnas numéricas
    assert df_clean['Age'].isnull().sum() == 0, 'Error: Siguen quedando nulos en Age'
    assert df_clean['MonthlyIncome'].isnull().sum() == 0, 'Error: Siguen quedando nulos en MonthlyIncome'
    assert df_clean['TrainingTimesLastYear'].isnull().sum() == 0, 'Error: Siguen quedando nulos en TrainingTimesLastYear'
    assert df_clean['YearsWithCurrManager'].isnull().sum() == 0, 'Error: Siguen quedando nulos en YearsWithCurrManager'

    # 4. Errata corregida
    assert 'Marreid' not in df_clean['MaritalStatus'].unique(), "Error: La errata 'Marreid' sigue existiendo"

    # 5. Tipos de datos correctos
    assert df_clean['Age'].dtype == 'int64', 'Error: Age no es int64'
    assert df_clean['MonthlyIncome'].dtype == 'int64', 'Error: MonthlyIncome no es int64'
    assert df_clean['Attrition'].dtype == 'Int64', 'Error: Attrition no es Int64'
    assert df_clean['OverTime'].dtype == 'Int64', 'Error: OverTime no es Int64'

    # 6. Binarias correctas
    attrition_vals = set(df_clean['Attrition'].dropna().unique())
    assert attrition_vals.issubset({0, 1}), 'Error: Attrition tiene valores fuera de {0, 1}'

    print('✅ ¡Todos los controles de calidad han pasado! El dataset está limpio y listo.')


# Ejecutamos la validación
validate(df_original, df_clean)

✅ ¡Todos los controles de calidad han pasado! El dataset está limpio y listo.


---
## Paso 10 — Guardamos el dataset limpio

In [31]:
# Guardamos el dataset limpio en un nuevo CSV sin el índice
df_clean.to_csv('hr_clean.csv', index=False)

print('✅ Archivo hr_clean.csv guardado con éxito.')
print(f'Dimensiones finales del dataset limpio: {df_clean.shape[0]} filas x {df_clean.shape[1]} columnas')

# Vista previa del resultado final
df_clean.head()

✅ Archivo hr_clean.csv guardado con éxito.
Dimensiones finales del dataset limpio: 1470 filas x 32 columnas


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,1,11,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,0,23,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,1,15,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,1,11,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,0,12,3,4,1,6,3,3,2,2,2,2


---
## Resumen de transformaciones aplicadas

| Paso | Acción | Columnas afectadas | Resultado |
|------|--------|--------------------|-----------|
| 1 | Eliminar columnas constantes | `EmployeeCount`, `Over18`, `StandardHours` | 35 → 32 columnas |
| 1 | Eliminar filas duplicadas | 4 filas duplicadas | 1474 → 1470 filas |
| 2 | Normalizar `JobRole` | `JobRole` | `strip()` + `title()` |
| 3 | Corregir errata | `MaritalStatus` | `'Marreid'` → `'Married'` |
| 4 | Nulos categóricos → moda | `Department`, `MaritalStatus`, `OverTime`, `BusinessTravel`, `EducationField` | 0 nulos |
| 5 | Nulos numéricos → mediana | `Age`, `JobSatisfaction`, `MonthlyIncome`, `TrainingTimesLastYear`, `YearsWithCurrManager` | 0 nulos |
| 6 | Tipos `float64` → `int64` | Las mismas 5 columnas numéricas | Sin decimales |
| 7 | Codificar binarias | `Attrition`, `OverTime` | `Yes/No` → `1/0` |

**Dataset resultante (`hr_clean.csv`): 1470 filas x 32 columnas — sin nulos, sin duplicados, tipos correctos.**